# Chapter 2 — The First Divergence

**Book alignment:** Debugging AI From First Principles, Chapter 2

**Question this notebook isolates:** A five-stage revenue pipeline emits one wrong total.
Do three unrelated defects (dropped refunds / double-counted shipments / stale promo)
produce that symptom at three *different* first-divergence stages — and does a
monotonicity check decide whether bisection is even safe?

Constructed illustration, deterministic, standard library only. No LM is called.

In [ ]:
# ingest -> clean -> aggregate -> discount -> render, as five pure functions.
# Each defect is a single keyword flag so the clean run is the known-good reference.
def ingest(raw):
    return [dict(o) for o in raw]

def clean(orders, *, drop_refunds=False):
    out = []
    for o in orders:
        if o["status"] == "voided":
            continue
        if o["status"] == "refunded":
            if drop_refunds:
                continue                                  # DEFECT A
            o = {**o, "amount": -abs(o["amount"])}         # spec: keep as a negative line
        out.append(o)
    return out

def aggregate(orders, *, per_shipment=False):
    revenue = round(sum(o["amount"] * (o["shipments"] if per_shipment else 1)   # DEFECT B
                        for o in orders), 2)
    return {"revenue": revenue, "orders": len(orders)}

def discount(agg, *, promo_rate=0.0):                      # current promo is 0.0
    return {**agg, "revenue": round(agg["revenue"] * (1 - promo_rate), 2)}     # DEFECT C: 0.05

def render(agg):
    return {"revenue_eur": f"{agg['revenue']:.2f}", "orders": agg["orders"]}

STAGES = ["ingest", "clean", "aggregate", "discount", "render"]

def _snap(x):
    if isinstance(x, list):
        return {"orders": len(x), "revenue": round(sum(o["amount"] for o in x), 2)}
    return dict(x)

def run(raw, *, defect=None):
    ck, x = {}, ingest(raw);                          ck["ingest"] = _snap(x)
    x = clean(x, drop_refunds=defect == "A");         ck["clean"] = _snap(x)
    x = aggregate(x, per_shipment=defect == "B");     ck["aggregate"] = _snap(x)
    x = discount(x, promo_rate=0.05 if defect == "C" else 0.0);  ck["discount"] = _snap(x)
    x = render(x);                                    ck["render"] = dict(x)
    return x, ck


FIXTURE = (
    [{"id": f"n{i}", "status": "ok", "amount": 100.00, "qty": 1, "shipments": 1} for i in range(13)]
    + [{"id": "s1", "status": "ok", "amount": 100.00, "qty": 1, "shipments": 2},
       {"id": "s2", "status": "ok", "amount": 100.00, "qty": 1, "shipments": 3}]
    + [{"id": f"r{i}", "status": "refunded", "amount": 15.00, "qty": 1, "shipments": 1} for i in range(12)]
    + [{"id": "r12", "status": "refunded", "amount": 16.20, "qty": 1, "shipments": 1},
       {"id": "r13", "status": "refunded", "amount": 16.20, "qty": 1, "shipments": 1}]
    + [{"id": "v1", "status": "voided", "amount": 999.0, "qty": 1, "shipments": 1},
       {"id": "v2", "status": "voided", "amount": 999.0, "qty": 1, "shipments": 1}]
)

## 1. One symptom, three causes, three wrong totals

15 real orders at €100, 14 refunds netting −€212.40, 2 voided. Intended revenue = €1,287.60.

In [ ]:
good_final, good_ck = run(FIXTURE)
print(f"intended revenue: €{good_final['revenue_eur']}\n")
for d in ("A", "B", "C"):
    final, _ = run(FIXTURE, defect=d)
    print(f"defect {d}: rendered €{final['revenue_eur']}  (wrong, by a different amount each time)")

assert good_final["revenue_eur"] == "1287.60"
assert {run(FIXTURE, defect=d)[0]["revenue_eur"] for d in "ABC"} == {"1500.00", "1587.60", "1223.22"}
print("\nfinal-output reasoning cannot separate A, B, C — they share the symptom class only")

## 2. Checkpoint the chain: the first mismatch owns the investigation

Compare each stage boundary against the known-good run, **in execution order**, and stop at
the first mismatch. Everything downstream of it is effect.

In [ ]:
def first_divergence(defect):
    _, ck = run(FIXTURE, defect=defect)
    for stage in STAGES:
        if ck[stage] != good_ck[stage]:
            return stage, ck[stage], good_ck[stage]
    return None

stage, obs, intended = first_divergence("A")
print(f"defect A first diverges at '{stage}':")
print(f"  observed  {obs}")
print(f"  intended  {intended}")

# downstream stages also mismatch — but they are quarantined until 'clean' is fixed
_, ckA = run(FIXTURE, defect="A")
downstream = [s for s in STAGES[STAGES.index(stage) + 1:] if ckA[s] != good_ck[s]]
assert stage == "clean" and downstream == ["aggregate", "discount", "render"]
print(f"  downstream (effects, not causes): {downstream}")

## 3. The same procedure isolates B and C to different stages

In [ ]:
firsts = {d: first_divergence(d)[0] for d in ("A", "B", "C")}
for d, s in firsts.items():
    print(f"defect {d}  ->  first divergence at '{s}'")

assert firsts == {"A": "clean", "B": "aggregate", "C": "discount"}
print("\nthree defects, one symptom, three first divergences -> three different fixes")

## 4. Bisection needs monotonicity — a masking stage breaks it

Binary search over a chain only works if "broken" stays broken. If a downstream stage
*partially cancels* an upstream error, the middle probe reads "nearly right" and the search
skips past the real break.

In [ ]:
monotone = [("ingest", 0.0), ("clean", 212.40), ("aggregate", 212.40),
            ("discount", 212.40), ("render", 212.40)]
masked   = [("ingest", 0.0), ("clean", 212.40), ("aggregate", 6.00),
            ("discount", 44.00), ("render", 212.40)]   # error cancelled at stage 3, re-emerges

def bisect_first_break(col, tol=10.0):
    lo, hi, probes = 0, len(col) - 1, []
    while lo < hi:
        mid = (lo + hi) // 2
        probes.append(col[mid][0])
        lo, hi = (mid + 1, hi) if abs(col[mid][1]) <= tol else (lo, mid)   # "within noise -> downstream"
    return col[lo][0], probes

def is_monotone(col):
    e = [abs(v) for _, v in col]
    return all(b >= a - 1e-9 for a, b in zip(e, e[1:]))

print("monotone column:", bisect_first_break(monotone), "  monotone?", is_monotone(monotone))
print("masked   column:", bisect_first_break(masked),   "  monotone?", is_monotone(masked))

assert bisect_first_break(monotone)[0] == "clean"       # correct
assert bisect_first_break(masked)[0] == "discount"      # WRONG — real break is 'clean'
assert is_monotone(monotone) and not is_monotone(masked)
print("\ncheck monotonicity first; a non-monotone column must be walked linearly, not bisected")

## 5. Within the stage: provenance of the wrong value

"The bug is in `clean`" is a location. "The bug is `clean` discarding these 14 rows whose
amounts sum to the exact error" is a diagnosis.

In [ ]:
def contributing_orders(orders, predicate):
    return [o for o in orders if predicate(o)]

guilty = contributing_orders(FIXTURE, lambda o: o["status"] == "refunded")
shortfall = round(sum(abs(o["amount"]) for o in guilty), 2)

a_rev = float(run(FIXTURE, defect="A")[0]["revenue_eur"])
good_rev = float(good_final["revenue_eur"])
print(f"{len(guilty)} refunded rows, |amount| sum = €{shortfall}")
print(f"defect A inflates revenue by €{round(a_rev - good_rev, 2)}")

assert len(guilty) == 14 and shortfall == 212.40
assert round(a_rev - good_rev, 2) == 212.40
print("the exact error is explained by a named set of input rows, not by a stage")

## What we earned

Final-output reasoning underdetermines the cause in any multi-stage execution: three
defects, one symptom, three first divergences. Ordered checkpoints localise in O(n);
bisection cuts that to O(log n) **only when the error column is monotone** — a masking
stage silently sends binary search to the wrong place. Inside the diverging stage, the
provenance of the wrong value bounds what to read.

**Notebook 03 / Chapter 3** adds the hygiene that keeps a fluent explanation — a comment, a
log line, a model's self-report — from contaminating the observations a checkpoint records.